In [1]:
import os
import re
import glob
from multiprocessing import Pool
from selectolax.parser import HTMLParser
from tqdm import tqdm

In [4]:
print('=' * 60)
print('SEC Filing HTML -> Text Converter (v2 - iXBRL fix)')
print('=' * 60)

print(f'\nScanning {RAW_HTML_PATH} ...')
all_files = []
for ext in ['*.html', '*.htm', '*.txt']:
    all_files.extend(glob.glob(os.path.join(RAW_HTML_PATH, '**', ext), recursive=True))
all_files = [f for f in all_files if os.path.isfile(f)]
print(f'Found {len(all_files)} files\n')

results = {'success': 0, 'skipped': 0, 'empty': 0,
           'xbrl_junk': 0, 'too_short': 0, 'failed': 0}

with Pool(processes=CPU_WORKERS, maxtasksperchild=50) as pool:
    for status in tqdm(
        pool.imap_unordered(convert_file, all_files, chunksize=10),
        total=len(all_files),
        desc='Converting'
    ):
        key = status if status in results else 'failed'
        results[key] += 1

print(f"\n{'=' * 60}")
print(f"SUCCESS (converted):     {results['success']}")
print(f"SKIPPED (already good):  {results['skipped']}")
print(f"XBRL JUNK (unreadable):  {results['xbrl_junk']}")
print(f"EMPTY (no content):      {results['empty']}")
print(f"TOO SHORT (<{MIN_OUTPUT_CHARS} chars): {results['too_short']}")
print(f"FAILED (errors):         {results['failed']}")
print(f"{'=' * 60}")

SEC Filing HTML -> Text Converter (v2 - iXBRL fix)

Scanning /home/ashish-varma-j/Desktop/SEC_AI_AGENT/data/raw_html ...
Found 38183 files



Converting:  14%|█▍        | 5400/38183 [36:45<3:43:08,  2.45it/s] Process ForkPoolWorker-5:



KeyboardInterrupt: 

In [ ]:
# ─── Verify output ────────────────────────────────────────────────────────────
md_files = glob.glob(os.path.join(MD_OUTPUT_PATH, '**', '*.md'), recursive=True)
good_files = [f for f in md_files if os.path.getsize(f) > MIN_OUTPUT_CHARS]

print(f'Total .md files on disk:  {len(md_files)}')
print(f'Files > {MIN_OUTPUT_CHARS} chars:        {len(good_files)}')

# Spot-check: verify no junk slipped through
xbrl_kw = ['xbrli:', 'us-gaap:', 'fasb.org', 'ix:nonfraction', 'http://xbrl']
junk_check = 0
import random
sample_set = random.sample(good_files, min(500, len(good_files)))
for f in sample_set:
    with open(f, 'r', errors='ignore') as fh:
        s = fh.read(1000)
    words = s.split()
    avg = sum(len(w) for w in words[:30]) / max(len(words[:30]), 1) if words else 0
    if avg > 20 or sum(1 for k in xbrl_kw if k in s) >= 2:
        junk_check += 1

print(f'\nSpot-check (500 random files): {junk_check} junk files found')
print('Quality looks good!' if junk_check == 0 else f'WARNING: {junk_check} bad files remain')

# Show 3 samples
print('\n--- Sample outputs ---')
for f in good_files[:3]:
    with open(f, 'r', errors='ignore') as fh:
        content = fh.read(300)
    print(f'\n{os.path.basename(f)}')
    print(content[:250])